In [7]:
import pandas as pd

df = pd.read_csv('RetailCleaned.csv', header=None)

df.columns = [
    'Invoice',
    'StockCode',
    'Description',
    'Quantity',
    'InvoiceDate',
    'Price',
    'Customer_ID',
    'Country',
    'SourceSheet'
]

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(df.shape)

df.info()

C:\Users\BRUNO\AppData\Local\Temp\ipykernel_1084\930405052.py:3: DtypeWarning: Columns (0: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('RetailCleaned.csv', header=None)


(1033036, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  str           
 8   SourceSheet  1033036 non-null  str           
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), str(4)
memory usage: 124.3+ MB


In [8]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY

rfm_data = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

snapshot_date = pd.to_datetime('2011-12-10')


rfm = rfm_data.groupby('Customer_ID').agg(

    Recency=('InvoiceDate',
             lambda x: (snapshot_date - x.max()).days),

    Frequency=('Invoice', 'nunique'),

    Monetary=('Quantity',
              lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())

).reset_index()

In [9]:
display(rfm.head(10))

,Customer_ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40
5,12351.0,375,1,300.93
6,12352.0,36,10,2849.84
7,12353.0,204,2,406.76
8,12354.0,232,1,1079.40
9,12355.0,214,2,947.61


In [10]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)


def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm['IsChurned'] = rfm['Recency'] > 90

In [ ]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID', how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)